# 🧹 Пайплайн препроцесингу тексту та токенізації (IMDb 50k)

**Дисципліна:** Глибинне навчання (ІПЗ-4)  
**Тема:** Movie Review Sentiment Analysis  
**Виконавець:** Денис (Data & Deployment Lead)  
**Мета:** Розробити повноцінний пайплайн очистки тексту (видалення HTML, очищення від шуму, пунктуації, стоп-слів, лематизація), токенізацію Keras, вирівнювання довжин (padding) та серіалізацію токенізатора для подальшого навчання нейромережі (LSTM) і використання у сервісі (Streamlit).

## 1. Імпорт бібліотек

In [ ]:
import os
import re
import pickle
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

# NLTK
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Scikit-Learn
from sklearn.model_selection import train_test_split

# Deep Learning Preprocessing (Keras)
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print(f"TensorFlow версія: {tf.__version__}")

## 2. Завантаження NLTK-ресурсів

In [ ]:
for res in ['stopwords', 'wordnet', 'omw-1.4']:
    try:
        nltk.download(res, quiet=True)
    except Exception as e:
        print(f"Помилка завантаження {res}: {e}")

# Отримуємо англійські стоп-слова, залишаючи заперечення (not, no, nor тощо)
base_stops = set(stopwords.words('english'))
negation_words = {'not', 'no', 'nor', 'neither', 'never', 'none'}
stop_words = base_stops - negation_words
lemmatizer = WordNetLemmatizer()

print(f"Кількість стоп-слів (збережено слова заперечення для коректного сентименту): {len(stop_words)}")

## 3. Завантаження даних

In [ ]:
DATA_PATH = os.path.join("..", "data", "IMDB Dataset.csv")

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
else:
    url = "https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv"
    df = pd.read_csv(url)
    os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
    df.to_csv(DATA_PATH, index=False)

print(f"Початковий розмір датасету: {df.shape}")
# Видалення дублікатів (виявлених під час EDA)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Розмір після видалення дублікатів: {df.shape}")

## 4. Пайплайн очищення тексту
Пайплайн включає:
1. Видалення HTML-розмітки за допомогою `BeautifulSoup`
2. Видалення URL та email-адрес
3. Видалення небуквених символів (пунктуації, цифр)
4. Переведення в нижній регістр
5. Фільтрацію стоп-слів з лематизацією

In [ ]:
def clean_text(text: str) -> str:
    """
    Комплексна функція очищення відгуку перед подачею в токенізатор.
    """
    # 1. Видалення HTML тегів
    text = BeautifulSoup(text, "html.parser").get_text()
    
    # 2. Видалення URL посилань
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # 3. Видалення небуквених символів
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # 4. Нижній регістр
    text = text.lower()
    
    # 5. Токенізація, фільтрація стоп-слів та лематизація
    tokens = text.split()
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words and len(word) > 1]
    
    return " ".join(cleaned_tokens)

# Тестування функції на зразку
sample_raw = "<br />The movie wasn't good at all! Terrible acting, worst 10/10 fail: https://site.com"
print("До очищення:  ", sample_raw)
print("Після очищення:", clean_text(sample_raw))

## 5. Обробка датасету та кодування цільової змінної

In [ ]:
# Застосовуємо очищення
print("Застосовуємо очищення до відгуків (це може зайняти до 1 хвилини)...")
df['cleaned_review'] = df['review'].apply(clean_text)

# Кодуємо цільову змінну: positive -> 1, negative -> 0
df['target'] = df['sentiment'].map({'positive': 1, 'negative': 0})

df[['review', 'cleaned_review', 'sentiment', 'target']].head()

## 6. Розподіл на вибірки: Train, Validation та Test

In [ ]:
X = df['cleaned_review'].values
y = df['target'].values

# 70% Train, 15% Validation, 15% Test зі збереженням балансу (stratify)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train розмір:      {len(X_train)} (70%)")
print(f"Validation розмір: {len(X_val)} (15%)")
print(f"Test розмір:       {len(X_test)} (15%)")

## 7. Токенізація Keras та паддінг (Padding)

На основі EDA висновків:
* `VOCAB_SIZE = 15000` (найчастіші слова, для OOV використовуємо токен `<OOV>`)
* `MAX_LEN = 200` (покриває переважну більшість відгуків без надлишкової пам'яті)

In [ ]:
VOCAB_SIZE = 15000
MAX_LEN = 200
OOV_TOKEN = "<OOV>"

# Навчаємо токенізатор лише на тренувальних даних!
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(X_train)

# Перетворюємо тексти в послідовності індексів
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Вирівнювання довжин за допомогою pad_sequences (post-padding)
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

print(f"Форма матриці X_train_pad: {X_train_pad.shape}")
print(f"Форма матриці X_val_pad:   {X_val_pad.shape}")
print(f"Форма матриці X_test_pad:  {X_test_pad.shape}")

## 8. Серіалізація токенізатора для моделі та Streamlit

In [ ]:
MODELS_DIR = os.path.join("..", "models")
os.makedirs(MODELS_DIR, exist_ok=True)
TOKENIZER_PATH = os.path.join(MODELS_DIR, "tokenizer.pkl")

# Зберігаємо токенізатор через pickle
with open(TOKENIZER_PATH, "wb") as f:
    pickle.dump(tokenizer, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Токенізатор успішно збережено в: {TOKENIZER_PATH}")

# Перевірка завантаження та інференс-пайплайну
with open(TOKENIZER_PATH, "rb") as f:
    loaded_tokenizer = pickle.load(f)

test_phrase = "This movie was an absolute masterpiece! Loved every minute."
cleaned_phrase = clean_text(test_phrase)
test_seq = loaded_tokenizer.texts_to_sequences([cleaned_phrase])
test_pad = pad_sequences(test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

print("Тестова фраза:   ", test_phrase)
print("Очищена фраза:   ", cleaned_phrase)
print("Паддований вектор (перші 15 токенів):", test_pad[0][:15])

## 9. Висновки етапу препроцесингу

1. **Очищення:** Повністю усунуто артефакти HTML (`<br />`), посилання, пунктуацію. Слова заперечення (`not`, `no`, `never`) збережено, оскільки вони критично змінюють полярність речення.
2. **Лематизація:** Застосовано `WordNetLemmatizer` для приведення форм слів до словникової основи.
3. **Train/Val/Test Split:** Створено вибірки 70% / 15% / 15% зі збереженням балансу класів.
4. **Токенізація та Padding:** Побудовано словник на 15 000 найважливіших слів, сформовано вектори довжиною 200.
5. **Готовність до DL:** Дані повністю готові для передачі Мирославу для побудови Embedding + LSTM архітектур у `notebooks/03_modeling_lstm.ipynb`.